# Verity: cloud run, part 3 (re-score, ~1 minute)

Run on the same server as parts 1 and 2. No new generation: it re-scores the LLM judges' cached answers with the corrected scoring.

In [17]:
# unpack the project code carried inside this notebook
import base64, io, zipfile, os
CODE = '...'
os.makedirs('/content/verity', exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE))).extractall('/content/verity')
os.chdir('/content/verity')
print(sorted(os.listdir('.')))

['data', 'experiments', 'figures', 'models', 'results', 'verity']


In [18]:
%cd /content/verity
!ls data/cache | grep llm_
!python -m experiments.e08_scale_up --only=rescore 2>&1 | grep -v -i warn | tail -12

/content
llm_local_Qwen_Qwen2.5-14B-Instruct.jsonl
llm_local_Qwen_Qwen2.5-32B-Instruct.jsonl
llm_local_Qwen_Qwen2.5-7B-Instruct.jsonl
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793131091.929929   15051 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793131091.930561   15051 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
qwen2.5-7b scifact_dev rescored macro-F1 0.728
qwen2.5-7b climate_fever_sample rescored macro-F1 0.593
qwen2.5-14b scifact_dev rescored macro-F1 0.753
qwen2.5-14b climate_fever_sample rescored macro-F1 0.543
qwen2.5-32b scifact_dev rescored macro-F1 0.752
qwen2.5-32b climate_fever_sample rescored macro-F1 0.55
RESU

In [19]:
# the results, printed so they are saved in this notebook
import json
r = json.load(open('/content/verity/results/e08_scale_up.json'))
print('RESULTS-JSON-START')
print(json.dumps(r))
print('RESULTS-JSON-END')

RESULTS-JSON-START
{"gpu": "NVIDIA A100-SXM4-80GB", "large_verifier": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli", "large_runs": [{"seed": 42, "calibration": {"n": 161, "accuracy": 0.8633540372670807, "macro_f1": 0.8604024459891972, "per_class": {"SUPPORTED": {"precision": 0.9206349206349206, "recall": 0.8405797101449275, "f1": 0.8787878787878788, "support": 69}, "REFUTED": {"precision": 0.9615384615384616, "recall": 0.7575757575757576, "f1": 0.847457627118644, "support": 33}, "NEI": {"precision": 0.7777777777777778, "recall": 0.9491525423728814, "f1": 0.8549618320610687, "support": 59}}, "macro_f1_ci95": [0.8001260475126473, 0.9126711692661376], "ece": 0.09713566784532915}, "scifact_dev": {"n": 300, "accuracy": 0.8333333333333334, "macro_f1": 0.8267327717506542, "per_class": {"SUPPORTED": {"precision": 0.9433962264150944, "recall": 0.8064516129032258, "f1": 0.8695652173913043, "support": 124}, "REFUTED": {"precision": 0.8545454545454545, "recall": 0.734375, "f1": 0.7899